# MPRF — Baselines supplémentaires (Top-k F1, clustering, aléatoire)
Placer d'abord les 4 fichiers de données dans `/content` (mêmes noms que pour les cellules précédentes), puis exécuter la cellule ci-dessous. Sortie : `mprf_baselines_raw.csv`.

In [ ]:
# ============================================================
# MPRF — Baselines supplémentaires d'élagage (R1.7)
#  - Top-k F1 : garde les k arbres au meilleur F1 sur la partition de validation (k = 5 et 10)
#  - Clustering : regroupe les arbres par similarité de prédictions (validation), garde le meilleur arbre
#                 de chaque groupe (k = 5 et 10 groupes) — élagage par clustering classique
#  - Aléatoire : k arbres tirés au hasard (k = 5 et 10) — témoin
#  Comparés à CSRF, OBP, IBEP et MPRF (alpha=0.6, k=10), mêmes plis, mêmes 5 seeds, même validation que cellD.
#  Tests de Wilcoxon appariés MPRF vs chaque nouvelle baseline, correction de Holm.
# Aucun fichier du papier n'est modifié. Google Colab — copier-coller directement
# ============================================================
import os, time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
from sklearn.cluster import AgglomerativeClustering
import warnings
warnings.filterwarnings("ignore")

# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]   # retirer ceux que tu ne veux pas relancer
SEEDS    = [0, 1, 2, 3, 4]          # seed 0 = exactement le protocole du papier
ALPHAS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
KS       = [5, 10, 15, 20]
ALPHA0, K0 = 0.6, 10                # configuration du papier
N_TREES, N_SPLITS = 50, 10
NESTED, NESTED_SEEDS, INNER_SPLITS = True, [0], 3   # alpha choisi par CV interne (3 plis) sur le train
N_BOOT   = 2000
OUT_DIR  = "."                      # les CSV sont écrits ici
FILES = {                           # mêmes fichiers que dans ton notebook
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
# └─────────────────────────────────────────────────────────────┘


# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]
SEEDS    = [0, 1, 2, 3, 4]
ALPHA0, K0 = 0.6, 10
KS_NEW   = [5, 10]
N_TREES, N_SPLITS, N_BOOT = 50, 10, 2000
OUT_DIR  = "."
FILES = {
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
# └─────────────────────────────────────────────────────────────┘

# ── Chargement (identique à ton notebook) ───────────────────
def load(name):
    path = FILES[name]
    if not os.path.exists(path):
        print(f"⚠️  {name}: fichier introuvable ({path}) → ignoré"); return None
    if name == "cervical":
        df = pd.read_csv(path); df.replace("?", np.nan, inplace=True)
        df = df.apply(pd.to_numeric, errors="coerce"); df.fillna(df.median(numeric_only=True), inplace=True)
        feats = [c for c in df.columns if c != "Biopsy"]
        return df[feats].values, df["Biopsy"].values.astype(int)
    if name == "hepatitis":
        df = pd.read_csv(path); df["Category"] = df["Category"].astype(str).str.strip()
        df["Category"] = df["Category"].map(lambda x: 0 if "Blood Donor" in x else (1 if "Hepatitis" in x else np.nan))
        df = df.dropna(subset=["Category"])
        feats = [c for c in df.columns if c not in ["Category", "Unnamed: 0"]]
        d = df[feats].copy()
        for col in d.columns:
            if d[col].dtype == object:
                le = LabelEncoder(); m = d[col].notna()
                d.loc[m, col] = le.fit_transform(d.loc[m, col].astype(str)); d[col] = pd.to_numeric(d[col], errors="coerce")
        d = d.apply(pd.to_numeric, errors="coerce"); d.fillna(d.median(numeric_only=True), inplace=True)
        return d.values, df["Category"].values.astype(int)
    if name == "pima":
        df = pd.read_csv(path); df.fillna(df.median(numeric_only=True), inplace=True)
        return df[[c for c in df.columns if c != "Outcome"]].values, df["Outcome"].values.astype(int)
    if name == "mammography":
        rows, data = [], False
        for line in open(path):
            line = line.strip()
            if line.lower() == "@data": data = True; continue
            if data and line:
                v = line.split(","); rows.append([float(t) for t in v[:6]] + [1 if v[6].strip().strip("'") == "1" else 0])
        a = np.array(rows); return a[:, :6], a[:, 6].astype(int)

# ── Métriques / score (mêmes formules que le notebook) ──────
def counts(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    return tp, fp, fn, tn

def f1_gm(tp, fp, fn, tn):
    den = 2 * tp + fp + fn
    f1 = 2 * tp / den if den > 0 else 0.0
    r1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    r0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return f1, np.sqrt(r1 * r0)

def metrics(y, p, proba):
    tp, fp, fn, tn = counts(y, p); f1, gm = f1_gm(tp, fp, fn, tn)
    return dict(f1_min=f1, gmean=gm,
                precision=tp / (tp + fp) if (tp + fp) > 0 else 0.0,
                sens=tp / (tp + fn) if (tp + fn) > 0 else 0.0,
                spec=tn / (tn + fp) if (tn + fp) > 0 else 0.0,
                fp=fp, fn=fn, auprc=average_precision_score(y, proba))

def score_vec(yv, pred_mat, alpha):          # pred_mat : (candidats, n_val) booléens
    pos = (yv == 1); neg = ~pos
    tp = (pred_mat & pos).sum(1); fp = (pred_mat & neg).sum(1)
    fn = (~pred_mat & pos).sum(1); tn = (~pred_mat & neg).sum(1)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    r1 = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    r0 = np.where(tn + fp > 0, tn / np.maximum(tn + fp, 1), 0.0)
    return alpha * f1 + (1 - alpha) * np.sqrt(r1 * r0)

# ── Sélection d'arbres (P = matrice (arbres, instances) des prédictions 0/1) ──
def MPRF(Pv, yv, k, alpha, init_seed, init="random"):
    n = Pv.shape[0]
    if init == "random":
        i0 = np.random.RandomState(init_seed).randint(0, n)
    else:                                         # "best" : meilleur arbre seul sur la validation
        i0 = int(np.argmax(score_vec(yv, Pv.astype(bool), alpha)))
    sel = [i0]; rem = [i for i in range(n) if i != i0]
    votes = Pv[i0].astype(float).copy()
    best = score_vec(yv, (votes >= 0.5)[None, :], alpha)[0]
    while len(sel) < k and rem:
        V = votes[None, :] + Pv[rem]
        sc = score_vec(yv, V >= (len(sel) + 1) / 2, alpha)
        j = int(np.argmax(sc))                    # 1er maximum = même règle que la boucle d'origine
        if sc[j] >= best:                         # plateau accepté
            votes = V[j]; sel.append(rem.pop(j))
            if sc[j] > best: best = sc[j]
        else:
            break
    return sel

def OBP(Pv, yv, max_size):
    n = Pv.shape[0]
    sc = [f1_gm(*counts(yv, Pv[i]))[0] for i in range(n)]
    order = np.argsort(sc)[::-1]; sel, best, votes = [], -np.inf, np.zeros(Pv.shape[1])
    for idx in order[:max_size]:
        v = votes + Pv[idx]; s = f1_gm(*counts(yv, (v >= (len(sel) + 1) / 2).astype(int)))[0]
        if s >= best: sel.append(idx); votes = v; best = s
    return sel if sel else [order[0]]

def IBEP(Pv, yv, max_size):
    n = Pv.shape[0]; err = (Pv != yv).astype(float); w = np.ones(len(yv)) / len(yv); sel = []
    for _ in range(max_size):
        b = int(np.argmin(err.dot(w)))
        if b in sel: break
        sel.append(b); w *= np.exp(err[b]); w /= w.sum()
    return sel if sel else [0]

def vote(P, sel):  return (P[sel].sum(0) >= len(sel) / 2).astype(int)

def forest(Xa, ya, seed, balanced):
    return RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced" if balanced else None,
                                  random_state=42 + seed, n_jobs=-1).fit(Xa, ya)

def tree_mats(rf, Xv, Xt):
    Pv = np.array([t.predict(Xv) for t in rf.estimators_])
    Pt = np.array([t.predict(Xt) for t in rf.estimators_])
    Qt = np.array([t.predict_proba(Xt)[:, 1] for t in rf.estimators_])
    return Pv, Pt, Qt

# ── Nouvelles baselines ─────────────────────────────────────
def topk_f1(Pv, yv, k):
    sc = np.array([f1_gm(*counts(yv, Pv[i]))[0] for i in range(Pv.shape[0])])
    return list(np.argsort(-sc, kind="stable")[:k])

def cluster_prune(Pv, yv, k):
    X = Pv.astype(float)
    try:    cl = AgglomerativeClustering(n_clusters=k, metric="hamming", linkage="average").fit(X)
    except TypeError: cl = AgglomerativeClustering(n_clusters=k, affinity="hamming", linkage="average").fit(X)
    sel = []
    for c in range(k):
        idx = np.where(cl.labels_ == c)[0]
        f = [f1_gm(*counts(yv, Pv[i]))[0] for i in idx]
        sel.append(int(idx[int(np.argmax(f))]))
    return sel

def random_k(n, k, seed):
    return list(np.random.RandomState(seed).choice(n, k, replace=False))

MAIN = f"MPRF a={ALPHA0:.1f} k={K0}"
NEW  = [f"{n} k={k}" for n in ("Top-k F1", "Cluster", "Random") for k in KS_NEW]

def run(name, X, y):
    rows = []; skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
    for seed in SEEDS:
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
            vi = np.random.RandomState(fold + 1000 * seed).choice(len(ytr), int(len(tr) * 0.2), replace=False)
            sub = np.setdiff1d(np.arange(len(ytr)), vi)
            Xv, yv, Xs, ys = Xtr[vi], ytr[vi], Xtr[sub], ytr[sub]
            iseed = 100 + fold + 1000 * seed
            rf = forest(Xs, ys, seed, True); Pv, Pt, Qt = tree_mats(rf, Xv, Xte)
            def add(method, pred, proba, size):
                d = metrics(yte, pred, proba); d.update(dataset=name, seed=seed, fold=fold, method=method, size=size); rows.append(d)
            add("CSRF", rf.predict(Xte), rf.predict_proba(Xte)[:, 1], N_TREES)
            for nm, sel in (("OBP", OBP(Pv, yv, N_TREES)), ("IBEP", IBEP(Pv, yv, N_TREES)), (MAIN, MPRF(Pv, yv, K0, ALPHA0, iseed))):
                add(nm, vote(Pt, sel), Qt[sel].mean(0), len(sel))
            for k in KS_NEW:
                for nm, sel in ((f"Top-k F1 k={k}", topk_f1(Pv, yv, k)), (f"Cluster k={k}", cluster_prune(Pv, yv, k)),
                                (f"Random k={k}", random_k(N_TREES, k, iseed))):
                    add(nm, vote(Pt, sel), Qt[sel].mean(0), len(sel))
        print(f"   {name}: seed {seed} terminé", flush=True)
    return pd.DataFrame(rows)

rng_b = np.random.RandomState(0)
def holm(ps):
    ps = np.array(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); run_max = 0.0; m = len(ps)
    for r, i in enumerate(o):
        run_max = max(run_max, min(1.0, (m - r) * ps[i])); adj[i] = run_max
    return adj

def report(R, name):
    D = R[R.dataset == name]
    per = D.groupby(["method", "seed"]).mean(numeric_only=True).reset_index()
    fold = D.groupby(["method", "fold"]).mean(numeric_only=True).reset_index()
    print("\n" + "=" * 100); print(f"  {name.upper()}   (moyenne sur plis et seeds ; ± = écart-type entre seeds)"); print("=" * 100)
    print(f"   {'méthode':<18} {'taille':>6} {'F1':>13} {'G-mean':>13} {'AUPRC':>13} {'Sens':>6} {'Spec':>6}")
    for m in ["CSRF", "OBP", "IBEP", MAIN] + NEW:
        p = per[per.method == m]; s = lambda c: f"{p[c].mean():.3f}±{p[c].std(ddof=0):.3f}"
        print(f"   {m:<18} {p['size'].mean():6.1f} {s('f1_min'):>13} {s('gmean'):>13} {s('auprc'):>13} {p['sens'].mean():6.3f} {p['spec'].mean():6.3f}")
    out = []
    for b in NEW:
        for met in ["f1_min", "gmean", "auprc"]:
            x = fold[fold.method == MAIN].sort_values("fold")[met].values; y_ = fold[fold.method == b].sort_values("fold")[met].values
            dd = x - y_; bm = [rng_b.choice(dd, len(dd)).mean() for _ in range(N_BOOT)]
            try: p = 1.0 if np.all(dd == 0) else wilcoxon(x, y_)[1]
            except Exception: p = np.nan
            out.append((b, met, dd.mean(), np.percentile(bm, 2.5), np.percentile(bm, 97.5), p))
    adj = holm([o[5] for o in out])
    print(f"\n   MPRF − baseline (Δ>0 : MPRF meilleur) ; IC95% bootstrap sur les 10 plis ; Wilcoxon ; Holm sur {len(out)} tests")
    print(f"   {'baseline':<16} {'métrique':<8} {'Δ moyen':>8} {'IC95%':>18} {'p brut':>8} {'p Holm':>8}")
    for o, a_ in zip(out, adj):
        print(f"   {o[0]:<16} {o[1]:<8} {o[2]:>+8.3f} [{o[3]:>+7.3f},{o[4]:>+7.3f}] {o[5]:>8.4f} {a_:>8.4f}")

t0 = time.time(); allres = []
for name in DATASETS:
    ld = load(name)
    if ld is None: continue
    X, y = ld; c = np.bincount(y)
    print(f"\n### {name} : {X.shape} | classes {c.tolist()} | IR={c[0]/c[1]:.1f}:1")
    R = run(name, X, y); allres.append(R); report(pd.concat(allres), name)
    pd.concat(allres).to_csv(os.path.join(OUT_DIR, "mprf_baselines_raw.csv"), index=False)
print(f"\nTerminé en {(time.time()-t0)/60:.1f} min — résultats bruts : mprf_baselines_raw.csv")
print("Contrôle : les lignes CSRF / OBP / IBEP / MPRF doivent retrouver les valeurs de cellD (5 seeds).")
